# Topic modelling stat.ML papers with BERTopic
What are statistical machine learning researchers working on, and how has that changed over the past year?

 This notebook uses BERTopic to find research topics in 7,624 arXiv papers from the stat.ML category (26 Nov 2025 – 24 Nov 2026), using each paper's title and cleaned abstract. The data was collected with `fetch_arxiv.py` and cleaned in `explore.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import umap
import os


## Loading the data
The data is loaded from data/papers_clean.csv, which was created in explore.ipynb. The file contains 7,624 papers listed in the stat.ML category (including cross-listed papers), submitted between 26/11/2025 and 24/11/2026. The text column is the title combined with the cleaned abstract. Cleaning removed LaTeX maths, line breaks and extra spaces, because maths symbols add noise to the embeddings and keywords. The title is keyword-dense and helps strengthen the signal in the longer abstract. The data is converted into a list because BERTopic expects a Python list of strings.

In [2]:
df = pd.read_csv("data/papers_clean.csv")
docs = df["text"].tolist()
print("Total documents:", len(docs))

Total documents: 7624


## How BERTopic works
BERTopic is a pipeline of four steps:

**1. Embeddings**
- Each paper becomes a 384-number vector, with papers that have similar meanings sitting close together.
- It captures meaning, so papers about "neural networks" and "deep models" end up close together even if they use different words.

**2. Dimensionality reduction (UMAP)**
- Shrinks the 384 dimensions to 5.
- Clustering in lots of dimensions works badly because the distances between points start to look similar.
- UMAP keeps nearby papers close; `n_neighbors=15` balances local and global structure.

**3. Clustering (HDBSCAN)**
- Density-based, so it finds regions where papers sit close together.
- The number of clusters is found automatically, not chosen in advance (unlike k-means).
- Papers that don't belong to any group are labelled -1 (outliers), which is about 38% of papers here.
- `min_topic_size` sets the smallest allowed cluster.

**4. Topic keywords (c-TF-IDF)**
- Treats all papers in a cluster as one document.
- Words that appear often in one cluster but rarely in others become that topic's keywords.
- c-TF-IDF only affects the labels, not which papers go into which cluster.


## Embedding
This section turns each of the 7,624 papers into a 384-number vector using all-MiniLM-L6-v2. Papers with similar meanings end up with similar vectors. This took roughly 10.5 minutes (239 batches) on my CPU, so the embeddings are saved to data/ and loaded from there afterwards instead of being recomputed. The model files are downloaded once and cached.

In [3]:
emb_path = "data/embeddings_minilm.npy"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(emb_path):
    embeddings = np.load(emb_path)
else:
    embeddings = model.encode(docs, show_progress_bar=True)
    np.save(emb_path, embeddings)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (7624, 384)


## Fitting the model
The embedding model is `all-MiniLM-L6-v2`, and the precomputed embeddings were passed in so it doesn't re-encode. The UMAP settings are `n_neighbors=15`, `n_components=5`, `min_dist=0.0`, `metric='cosine'` and `random_state=42`. These are BERTopic's default settings except `random_state`.

`random_state` makes the results reproducible. `min_dist` controls how tightly papers pack together; 0.0 packs them very tightly, which helps clustering. Cosine distance compares the direction of vectors, which is standard for text embeddings.

HDBSCAN uses the default settings, so the minimum topic size is 10.

The keywords use the c-TF-IDF defaults with no stop-word removal yet, which is why topic -1's words are "the", "and" and "of".

The run time was about 13 seconds because the slow embedding step was already done.

In [4]:
random_state=42
bertopic_model = BERTopic(embedding_model=model, umap_model=umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=random_state))
topics, probs = bertopic_model.fit_transform(docs, embeddings)


In [5]:
topic_info = bertopic_model.get_topic_info()
print("Number of topics:", len(topic_info))
print(topic_info.head(15))

Number of topics: 98
    Topic  Count                                            Name  \
0      -1   2898                                -1_the_and_of_to   
1       0    287              0_policy_reinforcement_learning_rl   
2       1    272              1_forecasting_series_time_forecast   
3       2    222                     2_regret_bandits_bandit_arm   
4       3    160     3_conformal_prediction_coverage_calibration   
5       4    152           4_clustering_cluster_clusters_mixture   
6       5    150  5_stochastic_gradient_convergence_optimization   
7       6    149    6_attention_transformers_transformer_softmax   
8       7    125                7_causal_discovery_dag_variables   
9       8    114               8_privacy_private_dp_differential   
10      9    106        9_bayesian_posterior_inference_amortized   
11     10     99              10_causal_treatment_effects_effect   
12     11     83               11_reasoning_grpo_training_policy   
13     12     79           

## First look at the topics
There are 97 topics plus one outlier group (-1). The outlier group has 2,898 papers, roughly 38%, which is quite high. The biggest topics are reinforcement learning (287), time-series forecasting (272), bandits (222), conformal prediction (160), clustering (152) and stochastic optimisation (150). The outlier keywords are made up of stop words like "the" and "of", so the keyword lists will need tuning with stop-word removal. The smallest topics only have 11 papers, close to the default `min_topic_size` of 10.

## Do the topics make sense?
To check quality, I read a random sample of 10 titles from 10 topics, including the outliers.

**Clean topics:** Topics 3 (conformal prediction), 7 (causal discovery) and 10 (treatment effects) are very consistent: almost every sampled title is clearly about that subject. Topics 7 and 10 are both causal, but they answer different questions (learning the causal graph vs estimating the size of an effect), so keeping them separate makes sense.

**Broader than their keywords:** Topic 0 is clearly reinforcement learning, but it mixes theory, robotics, LLM agents and game self-play. Topic 1 is labelled forecasting, but it also includes point processes, surrogate models and stock returns, so it is really a broader temporal and spatial data topic. Topic 11 (LLM reasoning with GRPO and chain-of-thought) overlaps with topic 0 because RL is now a common way to train LLMs.

**Messy topics:** Topic 32 is mostly representation learning (VAEs, ICA, disentanglement), but an MRI dataset paper slipped in. Topic 58 is a loose mix of loss-function theory, robustness and backdoors with no single theme. Topic 92 is small (11 papers) and grouped by application area (logistics, power systems) rather than by method.

**Near-duplicates:** Several topics cover the same broad area: diffusion and generative models (8 topics), optimisation (4), causal inference (4) and LLMs (6). 97 topics is more detail than is useful for a summary.

**Outliers:** Many papers in -1 clearly belong to existing topics. For example, two conformal prediction papers in the sample should be in topic 3. This suggests HDBSCAN is being cautious rather than most outliers being noise.

**Next step:** In the tuning section I will remove stop words, raise `min_topic_size` to merge near-duplicate topics, and use `reduce_outliers` to reassign misplaced papers.

In [6]:
pd.set_option("display.max_colwidth", None)
df["topic"] = topics

In [7]:
to_check = [-1, 0, 1, 3, 7, 10, 11, 32, 58, 92]

for topic_id in to_check:
    name = topic_info.loc[topic_info["Topic"] == topic_id, "Name"].iloc[0]
    count = (df["topic"] == topic_id).sum()
    print(f"Topic {topic_id} ({count} papers): {name}")
    sample = df[df["topic"] == topic_id].sample(n=min(10, count), random_state=42)
    display(sample[["title"]])

Topic -1 (2898 papers): -1_the_and_of_to


,title
358,Beyond Prediction Residuals: Conformal Regret Calibration for Robust Contextual Linear Optimization
4114,Soft Specialists: $α$-Rényi Ensembles for Uncertainty-Aware LLM Post-Training
1282,Data Reuse in Non-Stationary Learning
4490,On the Burden of Achieving Fairness in Conformal Prediction
5913,Harnessing Synthetic Data from Generative AI for Statistical Inference
3856,Sparse Functional Singular Value Decomposition for Biclustering and Triclustering Longitudinal Data
3030,Deep and Probabilistic Models for Gene Regulatory Network Inference
6558,Provable Target Sample Complexity Improvements as Pre-Trained Models Scale
1478,Latent Score-Based Bayesian Cramér-Rao Bound Estimation for High-Dimensional Imaging Systems
5208,"MCAnalysis: An Open-Source Package for Preprocessing, Modelling, and Visualisation of Menstrual Cycle Effects in Digital Health Data"


Topic 0 (287 papers): 0_policy_reinforcement_learning_rl


,title
224,Bellman Residual Bracketing: Resolving the Sample Complexity of Discounted and Average-Reward RL
6695,Cantelli Constrained Policy Optimization
4043,Physics-informed Goal-Conditioned Reinforcement Learning under Hybrid Contact Dynamics
5734,Delightful Policy Gradient
6143,Compositional Planning with Jumpy World Models
5218,Offline-Online Reinforcement Learning for Linear Mixture MDPs
2566,Offline Deep Q* Estimation with Diffusion Models
1979,Dual-Axis Policy Optimization for LLM Agents: Bayesian Feedback Attribution and Trajectory Mass Normalization
2916,Learning Ergodic Dynamical Systems from a Finite Trajectory
834,The Predictive--Optimization Gap in GuanDan Self-Play


Topic 1 (272 papers): 1_forecasting_series_time_forecast


,title
872,Divide-and-Conquer Neural Bayes Estimation for Spatial and Spatiotemporal Point Process Models
3568,TS-Fault: Benchmarking Time Series Forecasters Against Structural Faults
2443,GeoQ: Geometry-Aware Conditional Quantile Error Estimation for Scientific Surrogate Models
3852,Zero-Copy Semantic Contagion: An In-Memory Streaming Architecture for Evolving Attention Graphs
5655,Fast and Interpretable Autoregressive Estimation with Neural Network Backpropagation
4095,Decision-focused learning for optimal PV-Battery scheduling
5965,Adapting Time Series Foundation Models through Data Mixtures
1497,Shapley-based Structural Analysis of Neural Calibration for Stochastic Volatility Models
4797,Scalable inference of spatial regions and temporal signatures from time series
7213,Structured Event Representation and Stock Return Predictability


... (10 outputs left out)

Topic 58 (26 papers): 58_adversarial_augmentation_loss_class


,title
2660,"No Unique Minimizer, No Problem: On the Consistency of Robust Neural Classifiers"
5632,Minimax Generalized Cross-Entropy
190,Generative Data Augmentation Without External Data: An Empirical and Theoretical Study on Deep Networks
7183,Does the Data Processing Inequality Reflect Practice? On the Utility of Low-Level Tasks
3278,Fast SDP certification of neural networks : towards large multi-class datasets
3149,Statistically Undetectable Backdoors in Deep Neural Networks
4767,Ratio-based Loss Functions
689,Axiomatic Learning and Probabilistic Structures with first applications to variants of Arithmetic Machine Learning and beyond
7159,Fundamental Novel Consistency Theory: $H$-Consistency Bounds
2156,A Generalization of Amari's Bayesian Duality


Topic 92 (11 papers): 92_logistics_operational_supply_aigc


,title
3936,Decision-calibrated prediction sets for robust power system operations
78,A hybrid Graph-based Machine Learning Model for predicting Delivery Times in Supply Chain Logistics
6304,Learning to Approximate Uniform Facility Location via Graph Neural Networks
7124,Assured autonomy: How operations research powers and orchestrates generative AI systems
2221,A Constraint-Aware Generative Framework for Synthetic Origin-Destination Demand in Logistics Networks
382,Prescriptive Learning of Linear Policies with Feasibility Guarantees for Binary Decision-Making
4879,Joint Energy Management and Coordinated AIGC Workload Scheduling for Distributed Data Centers: A Diffusion-Aided Reward Shaping Approach
2924,FMOPF: Latent Flow Matching with Constraint-Aware Interaction Priors for AC Optimal Power Flow
4864,Middle-mile logistics through the lens of goal-conditioned reinforcement learning
2532,Supply Chain Analytics: A Data-Driven Approach


## Tuning

## Topics over time

## Findings

## Limitations and next steps